### 00 LIBRERIAS

In [56]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from glob import glob
# Aplicar configuraciones de visualización total de Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

from pathlib import Path
import glob

import warnings
warnings.filterwarnings("ignore")
import plotly.express as px

import func_negocio as func

### 01 EXTRACT

In [57]:
ruta_locales = r"input\dim\dim_locales.xlsx"
df_locales = pd.read_excel(ruta_locales)

In [58]:
ruta_puestos = r"input\dim\Mapeo_Puestos_Dotacion_Tipos.xlsx"
df_puestos = pd.read_excel(ruta_puestos)

In [59]:
ruta_secciones = r"input\dim\dim_secciones.xlsx"
df_secciones = pd.read_excel(ruta_secciones)

In [60]:
lista_excel = ["HrsReales_Export_202401-202506.xlsx", "HrsRealesLite_202512.xlsx"]
lista_csv = ["HrsRealesLite_202507.csv","HrsRealesLite_202508.csv", "HrsRealesLite_202509.csv", "HrsRealesLite_202510.csv", "HrsRealesLite_202511.csv", "HrsRealesLite_202601v2.csv",
             "HrsRealesLite_202602.csv", "HrsRealesLite_202603.csv"]
lista_csv2 = ["Hrs_reales_202604.csv", "Hrs_reales_202605.csv", "Hrs_reales_20260607.csv"]
ruta = r"input\bd_jeq"


In [61]:
rutas_completas = [os.path.join(ruta, archivo) for archivo in lista_excel]
dfs = [pd.read_excel(p) for p in rutas_completas]
df_jeq_1 = pd.concat(dfs, ignore_index=True)

In [62]:
rutas_completas = [os.path.join(ruta, archivo) for archivo in lista_csv]
dfs = [pd.read_csv(p, sep=",") for p in rutas_completas]
df_jeq_2 = pd.concat(dfs, ignore_index=True)

In [63]:
rutas_completas = [os.path.join(ruta, archivo) for archivo in lista_csv2]
dfs = [pd.read_csv(p, sep=";") for p in rutas_completas]
df_jeq_3 = pd.concat(dfs, ignore_index=True)

In [64]:
df_jeq_1["_Periodo"] = (df_jeq_1["FECHA"].dt.year).astype(str) + (df_jeq_1["FECHA"].dt.month).astype(str).str.zfill(2)

In [65]:
df_jeq_2["LOCAL"] = df_jeq_2["BD_HrsReal[LOCAL]"]
df_jeq_2["SECCION"] = df_jeq_2["BD_HrsReal[SECCION]"]
df_jeq_2["DNI"] = df_jeq_2["BD_HrsReal[DNI]"]
df_jeq_2["NOMBRE"] = df_jeq_2["BD_HrsReal[NOMBRE]"]
df_jeq_2["PUESTO"] = df_jeq_2["BD_HrsReal[PUESTO]"]
df_jeq_2["_Periodo"] = df_jeq_2["Tiempo[_Periodo01]"].astype(str).str[0:6]
df_jeq_2["HRS_REAL"] = df_jeq_2["[HRS_REAL]"]

df_jeq_2[['LOCAL', 'SECCION', 'DNI', 'NOMBRE', 'PUESTO', '_Periodo', 'HRS_REAL']].head(2)

,LOCAL,SECCION,DNI,NOMBRE,PUESTO,_Periodo,HRS_REAL
0,CFR PVEA SANTA CLARA,CAJAS,73257589,"CONDOR PONCE, FERNANDO WILSON",CFR CAJERO 5X2 2DO TURNO,202507,99.0
1,CFR PVEA SANTA CLARA,CAJAS,73615663,"HUAMAN DELA CRUZ, MARICRUZ",CFR CAJERO 5X2 2DO TURNO,202507,99.0


In [66]:
df_jeq_3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1244648 entries, 0 to 1244647
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype  
---  ------         --------------    -----  
 0   LOCAL          1244648 non-null  object 
 1   SECCION        1244648 non-null  object 
 2   DNI            1244648 non-null  int64  
 3   NOMBRE         1244648 non-null  object 
 4   PUESTO         1244648 non-null  object 
 5   FECHA          1244648 non-null  object 
 6   HRS_REAL       1244648 non-null  float64
 7   HRS_REAL_HHMM  1244648 non-null  object 
 8   CODIGO LOCAL   1244648 non-null  int64  
dtypes: float64(1), int64(2), object(6)
memory usage: 85.5+ MB


In [67]:
df_jeq_3["FECHA"] = pd.to_datetime(df_jeq_3["FECHA"], format="%d/%m/%Y")
df_jeq_3["_Periodo"] = (df_jeq_3["FECHA"].dt.year).astype(str) + (df_jeq_3["FECHA"].dt.month).astype(str).str.zfill(2)
df_jeq_3[['LOCAL', 'SECCION', 'DNI', 'NOMBRE', 'PUESTO', '_Periodo', 'HRS_REAL']].head(2)

,LOCAL,SECCION,DNI,NOMBRE,PUESTO,_Periodo,HRS_REAL
0,MAKRO SAN JUAN DE MIRAFLORES,ABARROTES,76766474,"LUNA RAMOS, LUCIA KARINA",MAKRO AUXILIAR DE ABARROTES,202604,8.0
1,MASS MS JASPES 20,MULTIFUNCIONAL,61215764,"ABARCA CUSTODIO, JUAN MANUEL",MASS OPERADOR DE TIENDA MASS,202604,7.9


In [68]:
df_jeq_final = pd.concat([df_jeq_1[['LOCAL', 'SECCION', 'DNI', 'NOMBRE', 'PUESTO','_Periodo', 'HRS_REAL']], df_jeq_2[['LOCAL', 'SECCION', 'DNI', 'NOMBRE', 'PUESTO', '_Periodo', 'HRS_REAL']],
                          df_jeq_3[['LOCAL', 'SECCION', 'DNI', 'NOMBRE', 'PUESTO', '_Periodo', 'HRS_REAL']]])

### 02 TRANSFORM

In [69]:
df_jeq_final.head(1)

,LOCAL,SECCION,DNI,NOMBRE,PUESTO,_Periodo,HRS_REAL
0,CFR PVEA AVENTURA SJL,CAJAS,71385932,"BUSTAMANTE FUENTES, DARLYNE MILAGROS",CFR CAJERO 6X1 2DO TURNO,202401,97.5


In [70]:
df_jeq_final1 = df_jeq_final.merge(df_locales[["nombre_Kr", "codigo_local", "Tienda"]], left_on="LOCAL", right_on="nombre_Kr", how="left")
df_jeq_final2 = df_jeq_final1[~df_jeq_final1["nombre_Kr"].isnull()]

In [71]:
df_jeq_final2

,LOCAL,SECCION,DNI,NOMBRE,PUESTO,_Periodo,HRS_REAL,nombre_Kr,codigo_local,Tienda
0,CFR PVEA AVENTURA SJL,CAJAS,71385932,"BUSTAMANTE FUENTES, DARLYNE MILAGROS",CFR CAJERO 6X1 2DO TURNO,202401,97.50,CFR PVEA AVENTURA SJL,1593,1593_Aventura SJL - PVH
1,CFR PVEA AREQUIPA,CAJAS,76293715,"CUSI LIPA, ISAAC RICARDO",CFR CAJERO 6X1 2DO TURNO,202401,97.50,CFR PVEA AREQUIPA,P104,P104_Arequipa - PVH
2,CFR PVEA AVENTURA SJL,CAJAS,70836619,"URQUIA AYBAR, YASSER",CFR CAJERO 6X1 2DO TURNO,202401,97.50,CFR PVEA AVENTURA SJL,1593,1593_Aventura SJL - PVH
3,CFR PVEA BOLICHERA,CAJAS,71053597,"LUNA SHUPINGAHUA, MARYCIELO MILAGROS",CFR CAJERO 6X1 2DO TURNO,202401,97.50,CFR PVEA BOLICHERA,P080,P080_Bolichera - PVH
4,CFR PVEA BOLICHERA,CAJAS,70404776,"ALARCON BACA, JOSE CARLOS",CFR CAJERO 6X1 2DO TURNO,202401,97.50,CFR PVEA BOLICHERA,P080,P080_Bolichera - PVH
...,...,...,...,...,...,...,...,...,...,...
1537856,SPO PVO JAEN,SALA,75075495,"LORENZO RAMIREZ, JHON WILLIAM",SPO REPRES. DE SERVICIOS 5X2 2DO TURNO,202606,4.50,SPO PVO JAEN,SO04,SO04_Jaen - PVO
1537857,SPO PVO JAEN,SALA,75075495,"LORENZO RAMIREZ, JHON WILLIAM",SPO REPRES. DE SERVICIOS 5X2 2DO TURNO,202606,4.50,SPO PVO JAEN,SO04,SO04_Jaen - PVO
1537858,SPO PVO JAEN,SALA,75159455,"BURGA MAJUAN, KATIA MILUSKA",SPO REPRES. DE SERVICIOS,202606,9.38,SPO PVO JAEN,SO04,SO04_Jaen - PVO
1537859,SPO PVO JAEN,SALA,75159455,"BURGA MAJUAN, KATIA MILUSKA",SPO REPRES. DE SERVICIOS,202606,10.35,SPO PVO JAEN,SO04,SO04_Jaen - PVO


In [72]:
df_jeq_final3  = df_jeq_final2.groupby(["_Periodo", "codigo_local", "Tienda", "PUESTO", "SECCION"])["HRS_REAL"].sum().reset_index()

In [73]:
def clean_text_series(series):
    # .str accede a las operaciones de cadena de forma vectorizada a nivel de C
    return (
        series.astype(str)
        .str.upper()
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("utf-8")
        .str.strip()
    )

In [74]:
df_puestos_clean = df_puestos[
    ["Departamento", "Nombre de Posición", "Grupo_Area"]
].copy()

# Normalizamos las llaves de cruce en el maestro
df_puestos_clean["Departamento"] = clean_text_series(
    df_puestos_clean["Departamento"]
)
df_puestos_clean["Nombre de Posición"] = clean_text_series(
    df_puestos_clean["Nombre de Posición"]
)

# Eliminar duplicados basándonos en las llaves (Equivale al FIRSTNONBLANK de DAX)
df_puestos_clean.drop_duplicates(
    subset=["Departamento", "Nombre de Posición"], inplace=True
)

In [75]:
df_jeq_final3_clean = df_jeq_final3.copy()

# Normalizamos las llaves de cruce en la tabla principal
df_jeq_final3_clean["SECCION"] = clean_text_series(
    df_jeq_final3_clean["SECCION"]
)
df_jeq_final3_clean["PUESTO"] = clean_text_series(df_jeq_final3_clean["PUESTO"])

In [76]:
df_jeq_final4 = df_jeq_final3_clean.merge(
    df_puestos_clean[["Departamento", "Nombre de Posición", "Grupo_Area"]],
    left_on=["SECCION", "PUESTO"],
    right_on=["Departamento", "Nombre de Posición"],
    how="left",
)
df_jeq_final4.rename(columns={"Grupo_Area": "_GrupoArea"}, inplace=True)


In [77]:
df_jeq_final4["_GrupoArea"].unique()

array(['Cajas', 'Inventarios', 'Almacén', 'Almacén Frescos', 'Abarrotes',
       'Frescos', 'Electro', 'Recepción', 'Multifuncional', 'Adm'],
      dtype=object)

In [78]:
df_secciones["TIPO"].unique()

array(['CAJAS', 'FRESCOS', 'ABARROTES', 'RECEPCIÓN', 'INVENTARIOS',
       'ALMACÉN FRESCOS', 'ALMACÉN', 'ELECTRO', 'SIN ELECTRO',
       'MULTIFUNCIONAL'], dtype=object)

In [79]:
df_secciones_clean = df_secciones.copy()
df_secciones_clean["TIPO"] = clean_text_series(df_secciones_clean["TIPO"])
df_secciones_clean.drop_duplicates(subset=["TIPO"], inplace=True)

df_jeq_final4_clean = df_jeq_final4.copy()
df_jeq_final4_clean["_GrupoArea"] = clean_text_series(df_jeq_final4_clean["_GrupoArea"])

df_jeq_final5 = df_jeq_final4_clean.merge( df_secciones_clean, left_on="_GrupoArea", right_on="TIPO", how="left")

df_jeq_final5["_Tipo"] = np.where(
    df_jeq_final5["_GrupoArea"] == "ADM",
    "Adm",
    np.where(
        df_jeq_final5["TIPO"].isna(), "adm", df_jeq_final5["TIPO"]
    ),
)

df_jeq_final5["_Tienda"] = df_jeq_final5["Tienda"]
df_jeq_final6 = df_jeq_final5.groupby(["_Periodo", "codigo_local", "_Tienda", "_Tipo", "_GrupoArea"])[ "HRS_REAL"].sum().reset_index()

In [80]:
periodo_dt = pd.to_datetime(df_jeq_final6["_Periodo"], format="%Y%m")
cant_dias_mes = periodo_dt.dt.days_in_month

factor_dia = 192 / 28
factor_mes = factor_dia * cant_dias_mes

# 4. Calcular el resultado final (_JEQv2)
df_jeq_final6["JEq"] = df_jeq_final6["HRS_REAL"] / factor_mes

In [81]:
df_jeq_final6["_Tipo"].unique()

array(['ABARROTES', 'ALMACEN', 'ALMACEN FRESCOS', 'CAJAS', 'ELECTRO',
       'FRESCOS', 'INVENTARIOS', 'RECEPCION', 'MULTIFUNCIONAL', 'Adm'],
      dtype=object)

### 03 LOAD

In [82]:
df_jeq_final6.to_parquet(r"output\JEq.parquet")